# Tutorial 02 — Primitives: from circuits to numbers

A `PrimitiveAlgorithm` declares **what you want to extract** from circuits.
You hand primitives to an engine; the engine returns one result per primitive.

The two workhorses:

| Primitive | Returns | How |
|---|---|---|
| `Sampler` | bitstring distribution (`SamplingDictionary`) | simulates and samples shots |
| `StateVector` | exact `float`/`complex` scalar | exact statevector contraction, no shots |

Shot-based *estimators* of the same scalars exist too (`TermwiseHadamardTest`,
`SWAPTest`, `PauliAveraging`, ...) — same interface, sampled instead of exact.
See `mwe_primitive_algorithms.ipynb` for the catalogue.

## 1. The target is inferred from what you pass

Primitives take up to three ingredients — `bra`, `operator`, `ket` — and infer
the quantity (their `target`) from which ones you supply:

* `ket` only → **sampling**
* `bra`, `ket` → **overlap** $\langle \mathrm{bra}|\mathrm{ket}\rangle$
* `bra=ket`, `operator` → **expectation value** $\langle \psi|H|\psi\rangle$
* `bra`, `operator`, `ket` → **transition amplitude** $\langle \mathrm{bra}|H|\mathrm{ket}\rangle$

In [ ]:
import numpy as np
from qarp.operators import QubitOperator
from qarp.blocks import ComputationalBasisStateBlock, HEABlock, HnBlock
from qarp.algorithms import Sampler, StateVector

psi = HEABlock(3, 2, True, True, True, False).build()
psi_01 = psi.set_symbols({s: 0.1 for s in psi.symbols}).build()
plus = HnBlock(n_qubits=3).build()

H = QubitOperator("Z0 Z1", -1.0) + QubitOperator("X2", 0.5)

for prim in (
    Sampler(ket=psi_01),
    StateVector(bra=plus, ket=psi_01),
    StateVector(bra=psi_01, operator=H, ket=psi_01),
    StateVector(bra=plus, operator=H, ket=psi_01),
):
    prim.build()
    print(f"{type(prim).__name__:12s} -> {prim.target}")

## 2. Exact expectation values with `StateVector`

In [ ]:
from qarp.engines import QarpEngine

expval = StateVector(bra=psi, operator=H, ket=psi)   # symbolic ansatz: values at run time

engine = QarpEngine()
engine.build([expval])

values = dict(zip(psi.symbols, np.linspace(0.0, 1.0, len(psi.symbols))))
energy = engine.run(values)[0]
print("<psi|H|psi> =", energy)
print("stored on the primitive too:", expval.result)

Notes:

* The operator is a `qarp.operators.QubitOperator` (openfermion-compatible API);
  its qubit indices are qarpx qubit indices, and `op.sparse_matrix()` realizes it in
  the same LSB convention as every qarpx statevector — no conversion anywhere.
* Only when you bring in an *external* MSB-first matrix or statevector (openfermion,
  cirq, pennylane) do you convert, with `qarp.endianness.msb_to_lsb_matrix` /
  `..._statevector` (eigenvalues need no conversion — they don't depend on bit order).

## 3. Overlaps

In [ ]:
overlap = StateVector(bra=plus, ket=psi_01)
engine = QarpEngine()
engine.build([overlap])
print("<+++|psi(0.1)> =", engine.run()[0])

## 4. Sampling, and shot-based estimation

`Sampler` returns the measured distribution; `n_shots` set on the primitive
overrides the engine default. For a *shot-based estimate of an expectation
value* (what real hardware gives you), use `TermwiseHadamardTest` or
`PauliAveraging` — same `bra/operator/ket` interface as `StateVector`:

In [ ]:
from qarp.algorithms import TermwiseHadamardTest

exact = StateVector(bra=psi_01, operator=H, ket=psi_01)
estimated = TermwiseHadamardTest(bra=psi_01, operator=H, ket=psi_01, n_shots=20_000)

engine = QarpEngine(seed=7)
engine.build([exact, estimated])      # one engine, several primitives
res = engine.run()
print("exact:    ", res[0])
print("estimated:", res[1])

`Sampler` returns a `SamplingDistribution`: it reads like a dict keyed by
LSB-first bit tuples over `measured_qubits`.  With `n_shots=qarp.EXACT` it is
the exact Born distribution instead of a draw.  For wide registers, read the
two aligned arrays instead of iterating tuples: `outcomes` holds the integer
`Σ_i b_i 2^i` of each key, `probabilities` its probability.

In [ ]:
import qarp
from qarp.blocks import SimpleBlock

ghz = SimpleBlock(3, name="ghz")
ghz.h(0)
ghz.cx(0, 1)
ghz.cx(1, 2)

sampler = Sampler(ket=ghz, n_shots=qarp.EXACT)
engine = QarpEngine()
engine.build([sampler])
dist = engine.run()[0]

print(dist)                     # ≈ {(0, 0, 0): 0.5, (1, 1, 1): 0.5}
print(dist[(1, 1, 1)])          # dict-style lookup
print(dist.outcomes)            # [0 7]: the integers of (0, 0, 0) and (1, 1, 1)
print(dist.probabilities)       # ≈ [0.5 0.5]

## Choosing between them

* **Developing / debugging an algorithm** → `StateVector`. Deterministic, fast,
  differentiable (tutorial 03 shows gradients).
* **Modelling what hardware would measure** → `Sampler` / Hadamard-test family
  with a finite `n_shots` and a seeded engine.

## Poke at it

* `prim.target`, `prim.n_shots`, `prim.result` (and `result_list` on termwise
  estimators — one entry per Pauli term)
* `prim.compiled_circuits` after `engine.build` — the transpiled command
  streams the engine will actually simulate

**Next:** tutorial_03_engines — the compile-once / run-many split, seeds,
batching, and gradients.